<a href="https://colab.research.google.com/github/gideonjeffrey/texas_voter_suspense_analysis/blob/main/historical_tx_voter_registration_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from io import StringIO
import plotly.express as px

### Extract historical data

In [ ]:
historical_reg_url = "https://www.sos.state.tx.us/elections/historical/vrfig.shtml"
historical_data = requests.get(historical_reg_url)

In [ ]:
soup = BeautifulSoup(historical_data.content, "html.parser")

specific_election_paths = []

for link in soup.select("#content li a"):
    specific_election_paths.append(link["href"])

In [ ]:
specific_election_paths

['mar2026.shtml',
 'nov2026.shtml',
 'jan2025.shtml',
 'nov2025.shtml',
 'jan2024.shtml',
 'mar2024.shtml',
 'nov2024.shtml',
 'jan2023.shtml',
 'nov2023.shtml',
 'jan2022.shtml',
 'march2022.shtml',
 'nov2022.shtml',
 'jan2021.shtml',
 'nov2021.shtml',
 'jan2020.shtml',
 'mar2020.shtml',
 'nov2020.shtml',
 'jan2019.shtml',
 'nov2019.shtml',
 'jan2018.shtml',
 'mar2018.shtml',
 'nov2018.shtml',
 'jan2017.shtml',
 'nov2017.shtml',
 'jan2016.shtml',
 'mar2016.shtml',
 'nov2016.shtml',
 'jan2015.shtml',
 'nov2015.shtml',
 'jan2014.shtml',
 'mar2014.shtml',
 'nov2014.shtml',
 'jan2013.shtml',
 'nov2013.shtml',
 'jan2012.shtml',
 'may2012.shtml',
 'nov2012.shtml',
 'jan2011.shtml',
 'nov2011.shtml',
 'jan2010.shtml',
 'mar2010.shtml',
 'nov2010.shtml',
 'jan2009.shtml',
 'nov2009.shtml',
 'jan08.shtml',
 'mar08.shtml',
 'nov2008.shtml',
 'jan07.shtml',
 'nov07.shtml',
 'jan06.shtml',
 'mar06.shtml',
 'nov06.shtml',
 'jan05.shtml',
 'nov05.shtml',
 'jan04.shtml',
 'mar04.shtml',
 'nov04.shtm

In [ ]:
base_url = "https://www.sos.state.tx.us/elections/historical/"

election_dfs = []

for path in specific_election_paths:
  election_url = base_url + path
  election_response = requests.get(election_url)
  election_soup = BeautifulSoup(election_response.text, "html.parser")

  h1 = election_soup.find("h1").text
  year_and_month = " ".join(h1.split(sep=' ')[0:2])
  data_date = datetime.strptime(year_and_month, "%B %Y").date()

  election_table = election_soup.find("table", class_="norm")

  dfs = pd.read_html(StringIO(str(election_table)))
  election_df = dfs[0]
  election_df['Election Month'] = data_date

  election_dfs.append(election_df)

  print(data_date, ' extracted')

# for election in specific elections: request from page, beautiful soup, html parse table to Pandas DF

2026-03-01  extracted
2026-11-01  extracted
2025-01-01  extracted
2025-11-01  extracted
2024-01-01  extracted
2024-03-01  extracted
2024-11-01  extracted
2023-01-01  extracted
2023-11-01  extracted
2022-01-01  extracted
2022-03-01  extracted
2022-11-01  extracted
2021-01-01  extracted
2021-11-01  extracted
2020-01-01  extracted
2020-03-01  extracted
2020-11-01  extracted
2019-01-01  extracted
2019-11-01  extracted
2018-01-01  extracted
2018-03-01  extracted
2018-11-01  extracted
2017-01-01  extracted
2017-11-01  extracted
2016-01-01  extracted
2016-03-01  extracted
2016-11-01  extracted
2015-01-01  extracted
2015-11-01  extracted
2014-01-01  extracted
2014-03-01  extracted
2014-11-01  extracted
2013-01-01  extracted
2013-11-01  extracted
2012-01-01  extracted
2012-05-01  extracted
2012-11-01  extracted
2011-01-01  extracted
2011-11-01  extracted
2010-01-01  extracted
2010-03-01  extracted
2010-11-01  extracted
2009-01-01  extracted
2009-11-01  extracted
2008-01-01  extracted
2008-03-01

### Standardize format of all datasets

In [ ]:
column_dict = {}

for df in election_dfs:
  for column in df.columns:
    column_dict[column] = column_dict.get(column, 0) + 1

In [ ]:
column_dict

{'County Name': 44,
 'Precincts': 60,
 'Voter Registration': 45,
 'Suspense Voters': 68,
 'Non-Suspense Voters': 46,
 'Election Month': 83,
 'County': 39,
 'Suspense': 5,
 'Non Suspense Voters': 18,
 'Total': 17,
 'Precinct': 2,
 'Registered Voters': 2,
 'Pct': 3,
 'Pcts': 8,
 'Non Sususpense Voters': 1,
 'Non Sus Voters': 1,
 'Number Of Precincts': 8,
 'NonSuspense Voters': 6,
 'Total Registered Voters': 18,
 'Non Suspense': 1,
 'Effective Date': 18,
 'Number of Non Suspense Voters': 1,
 'Total Voter Registration': 1,
 'Number of Precincts': 1}

In [ ]:
column_mapping = {
 'Voter Registration': 'Total Registered Voters',
 'County': 'County Name',
 'Suspense': 'Suspense Voters',
 'Non Suspense Voters': 'Non-Suspense Voters',
 'Total': 'Total Registered Voters',
 'Registered Voters': 'Total Registered Voters',
 'Non Sususpense Voters': 'Non-Suspense Voters',
 'Non Sus Voters': 'Non-Suspense Voters',
 'NonSuspense Voters': 'Non-Suspense Voters',
 'Non Suspense': 'Non-Suspense Voters',
 'Number of Non Suspense Voters': 'Non-Suspense Voters',
 'Total Voter Registration': 'Total Registered Voters',
 'Precinct': 'Precincts',
 'Pct': 'Precincts',
 'Pcts': 'Precincts',
 'Number Of Precincts': 'Precincts',
 'Number of Precincts': 'Precincts'
}
for df in election_dfs:
  df.rename(columns=column_mapping, inplace=True)

In [ ]:
column_dict_revised = {}

for df in election_dfs:
  for column in df.columns:
    column_dict_revised[column] = column_dict_revised.get(column, 0) + 1

column_dict_revised

{'County Name': 83,
 'Precincts': 82,
 'Total Registered Voters': 83,
 'Suspense Voters': 73,
 'Non-Suspense Voters': 74,
 'Election Month': 83,
 'Effective Date': 18}

In [ ]:
missing_data_list = [df['Election Month'][0] for df in election_dfs if "Suspense Voters" not in df.columns and "Non-Suspense Voters" in df.columns ]

In [ ]:
missing_data_list

[datetime.date(1995, 11, 1)]

In [ ]:
combined_df = pd.DataFrame(columns=['Election Month', 'County Name', 'Total Registered Voters', 'Suspense Voters', 'Non-Suspense Voters', 'Suspense Percentage'])

In [ ]:
for df in election_dfs:
  if "Non-Suspense Voters" in df.columns:
    if "Suspense Voters" not in df.columns:
      df["Suspense Voters"] = df["Total Registered Voters"] - df["Non-Suspense Voters"]
    df["Suspense Voters"] = pd.to_numeric(df["Suspense Voters"], errors='coerce').fillna(0).astype(int)
    df["Suspense Percentage"] = (df["Suspense Voters"] / df["Total Registered Voters"] * 100.0).round(2)
    df["County Name"] = df["County Name"].str.upper()
    df["Election Month"] = pd.to_datetime(df["Election Month"])
    combined_df = pd.concat(
      [
        combined_df,
        df[['Election Month', 'County Name', 'Total Registered Voters', 'Suspense Voters', 'Non-Suspense Voters', 'Suspense Percentage']],
      ], ignore_index=True
    )


/tmp/ipykernel_1703/3984382715.py:9: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



In [ ]:
combined_df.shape

(18870, 6)

In [ ]:
total_records_by_county_name = combined_df["County Name"].value_counts().reset_index(name='count')

In [ ]:
total_records_by_county_name[total_records_by_county_name['count'] < 74]

,County Name,count
248,MCLENNAN,73
249,THROCKMORTON,73
250,FAYETTE,73
251,COLLINGSWORTH,72
252,FORT BEND,69
253,LASALLE,56
254,STATEWIDE TOTAL,40
255,TOTAL,33
256,LA SALLE,18
257,FT. BEND,5


Need to standardize county names; and we'll use "Statewide Total" for everything that's a total row.

In [ ]:
combined_df["County Name"] = combined_df["County Name"].replace(
    {
        'MCLENNEN': 'MCLENNAN',
        'COLLINSWORTH': 'COLLINGSWORTH',
        'COLLINGSWORT': 'COLLINGSWORTH',
        'THROCKMORTO': 'THROCKMORTON',
        'FAYETTE COUNTY': 'FAYETTE',
        'FT. BEND': 'FORT BEND',
        'LASALLE': 'LA SALLE',
        'TOTAL': 'STATEWIDE TOTAL',
        'GRAND TOTAL': 'STATEWIDE TOTAL'
    }
)

In [ ]:
#test again
total_records_by_county_name = combined_df["County Name"].value_counts().reset_index(name='count')
total_records_by_county_name[total_records_by_county_name['count'] < 74]

,County Name,count


### Create state-level and county-level datasets



In [ ]:
state_df = combined_df[combined_df['County Name'] == 'STATEWIDE TOTAL'].drop(columns=["County Name"])

In [ ]:
state_df.shape

(74, 5)

In [ ]:
state_df.sort_values('Election Month', ascending=False)

,Election Month,Total Registered Voters,Suspense Voters,Non-Suspense Voters,Suspense Percentage
509,2026-11-01,19041684,1763730,17277954,9.26
254,2026-03-01,18657918,1228918,17429000,6.59
1019,2025-11-01,18488166,1138186,17349980,6.16
764,2025-01-01,18258249,1159815,17098434,6.35
1784,2024-11-01,18623931,2012853,16611078,10.81
...,...,...,...,...,...
18104,1997-11-01,11066181,923992,10142189,8.35
17849,1997-01-01,10648846,924302,9723671,8.68
18614,1996-11-01,10540678,989487,9551191,9.39
18359,1996-01-01,9546658,587549,8959217,6.15


In [ ]:
county_df = combined_df[combined_df['County Name'] != 'STATEWIDE TOTAL']

In [ ]:
county_df.shape

(18796, 6)

In [ ]:
state_df.dtypes

,0
Election Month,datetime64[ns]
Total Registered Voters,object
Suspense Voters,object
Non-Suspense Voters,object
Suspense Percentage,float64


In [ ]:
### Quick visual examination of datasets

In [ ]:
fig = px.line(state_df.sort_values("Election Month"), x="Election Month", y="Suspense Percentage", title='Statewide Percentage of Suspense Voters, 1995-present')
fig.show()

In [ ]:
mclennan_fig = px.line(county_df[county_df['County Name'] == 'MCLENNAN'].sort_values("Election Month"), x="Election Month", y="Suspense Percentage", title='McLennan County Percentage of Suspense Voters, 1995-present')
mclennan_fig.show()

In [ ]:
reg_total_fig = px.line(state_df.sort_values("Election Month"), x="Election Month", y="Total Registered Voters", title='Registered Voters, 1995-present')
reg_total_fig.show()

In [ ]:
dallas_fig = px.line(county_df[county_df['County Name'] == 'DALLAS'].sort_values("Election Month"), x="Election Month", y="Suspense Percentage", title='Dallas County Percentage of Suspense Voters, 1995-present')
dallas_fig.show()

### Add FIPS codes and prepare for export

In [ ]:
state_df.columns = state_df.columns.str.upper().str.replace(' ', '_').str.replace('-', '_')

In [ ]:
county_df.columns = county_df.columns.str.upper().str.replace(' ', '_').str.replace('-', '_')

In [ ]:
fips_df = pd.read_csv('/content/FIPS_TEXAS_COUNTIES.csv')

In [ ]:
county_df.columns

Index(['ELECTION_MONTH', 'COUNTY_NAME', 'TOTAL_REGISTERED_VOTERS',
       'SUSPENSE_VOTERS', 'NON_SUSPENSE_VOTERS', 'SUSPENSE_PERCENTAGE'],
      dtype='object')

In [ ]:
fips_df.columns

Index(['COUNTY_NAME', 'FIPS_CODE'], dtype='object')

In [ ]:
county_df = county_df.merge(fips_df, how='left', on="COUNTY_NAME")

In [ ]:
county_df.shape

(18796, 7)

In [ ]:
county_df.head()

,ELECTION_MONTH,COUNTY_NAME,TOTAL_REGISTERED_VOTERS,SUSPENSE_VOTERS,NON_SUSPENSE_VOTERS,SUSPENSE_PERCENTAGE,FIPS_CODE
0,2026-03-01,ANDERSON,30678,1512,29166,4.93,48001
1,2026-03-01,ANDREWS,10637,524,10113,4.93,48003
2,2026-03-01,ANGELINA,55445,2247,53198,4.05,48005
3,2026-03-01,ARANSAS,19411,1441,17970,7.42,48007
4,2026-03-01,ARCHER,7005,197,6808,2.81,48009


In [ ]:
county_df[county_df['FIPS_CODE'].isna()]

,ELECTION_MONTH,COUNTY_NAME,TOTAL_REGISTERED_VOTERS,SUSPENSE_VOTERS,NON_SUSPENSE_VOTERS,SUSPENSE_PERCENTAGE,FIPS_CODE


In [ ]:
state_df.to_csv('tx_statewide_suspense_voter_history.csv', index=False)

In [ ]:
county_df.to_csv('tx_county_suspense_voter_history.csv', index=False)